## 1. Environment setup

This cell imports required libraries, mounts Drive in Colab, and prepares the output directories.

In [9]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

IN_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path.cwd()

OUTPUTS_ROOT = DRIVE_ROOT / "Outputs"
DATASETS_ROOT = DRIVE_ROOT / "Datasets"
GENERATED_DATASETS_ROOT = OUTPUTS_ROOT / "generated_datasets"

OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)
GENERATED_DATASETS_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = OUTPUTS_ROOT / RUN_ID
FIGURES_DIR = RUN_DIR / "figures"
TABLES_DIR = RUN_DIR / "tables"
OTHERS_DIR = RUN_DIR / "others"

for d in [RUN_DIR, FIGURES_DIR, TABLES_DIR, OTHERS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = OUTPUTS_ROOT / "outputs_summary.txt"

print(f"IN_COLAB        : {IN_COLAB}")
print(f"DRIVE_ROOT      : {DRIVE_ROOT}")
print(f"DATASETS_DIR    : {DATASETS_ROOT}")
print(f"RUN_DIR         : {RUN_DIR}")
print(f"SUMMARY PATH    : {SUMMARY_PATH}")

IN_COLAB        : False
DRIVE_ROOT      : F:\MBZUAI UAE\7-CMES-SI\GitHub-v2
DATASETS_DIR    : F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Datasets
RUN_DIR         : F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819
SUMMARY PATH    : F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\outputs_summary.txt


## 2. Logging and utility functions

This cell defines helper functions for logging, saving outputs, path resolution, and metric computation.

In [10]:
def log_line(text: str = ""):
    print(text)
    with open(SUMMARY_PATH, "a", encoding="utf-8") as f:
        f.write(str(text) + "\n")

def log_section(title: str):
    sep = "=" * 100
    log_line("\n" + sep)
    log_line(title)
    log_line(sep)

def log_kv(key: str, value):
    log_line(f"{key}: {value}")

def save_csv(df: pd.DataFrame, name: str):
    path = TABLES_DIR / name
    df.to_csv(path, index=False)
    log_line(f"Saved table: {path}")
    return path

def save_json(obj, name: str):
    path = OTHERS_DIR / name
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
    log_line(f"Saved json: {path}")
    return path

def save_figure(name: str):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    log_line(f"Saved figure: {path}")
    plt.close()
    return path

def resolve_existing_path(filename=None, subdir=None, full_path=None, recursive_search=True):
    tried = []
    if full_path:
        p = Path(full_path)
        tried.append(p)
        if p.exists():
            return p

    base_paths = [DRIVE_ROOT, Path("/content"), Path("/mnt/data"), Path(".")]
    for base in base_paths:
        if filename and subdir:
            p = base / subdir / filename
            tried.append(p)
            if p.exists():
                return p
        if filename:
            p = base / filename
            tried.append(p)
            if p.exists():
                return p

    if recursive_search and filename and DRIVE_ROOT.exists():
        hits = list(DRIVE_ROOT.rglob(filename))
        if hits:
            return hits[0]

    searched = "\n".join(str(x) for x in tried)
    raise FileNotFoundError(f"Could not find the requested file. Searched:\n{searched}")

def compute_metrics(y_true, y_pred, y_score):
    acc = accuracy_score(y_true, y_pred)
    bal = balanced_accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    try:
        roc = roc_auc_score(y_true, y_score)
    except Exception:
        roc = np.nan

    try:
        pr = average_precision_score(y_true, y_score)
    except Exception:
        pr = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

    return {
        "accuracy": float(acc),
        "balanced_accuracy": float(bal),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "roc_auc": float(roc) if not pd.isna(roc) else np.nan,
        "pr_auc": float(pr) if not pd.isna(pr) else np.nan,
        "fpr": float(fpr) if not pd.isna(fpr) else np.nan,
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }

def single_feature_auc_table(df, target_col, candidate_features):
    rows = []
    y = df[target_col].astype(int).values
    for col in candidate_features:
        x = df[col]
        if pd.api.types.is_numeric_dtype(x):
            score = x.fillna(x.median()).values
        else:
            score = pd.factorize(x.astype(str))[0]
        try:
            auc = roc_auc_score(y, score)
            if auc < 0.5:
                auc = 1 - auc
        except Exception:
            auc = np.nan
        rows.append({"feature": col, "roc_auc": auc})
    return pd.DataFrame(rows).sort_values("roc_auc", ascending=False)

## 3. Configuration

This cell defines the dataset paths, generation settings, model settings, fusion settings, and graph settings.

In [11]:
CONFIG = {
    "random_state": 42,

    # Cyber dataset
    "cyber_full_path": None,
    "cyber_filename": "ton_iot.csv",
    "cyber_subdir": "Datasets/TON_IoT",
    "cyber_target_col": "label",
    "cyber_context_col": "type",
    "cyber_test_size": 0.30,
    "strict_identifier_exclusion": True,

    # Video dataset
    "realistic_video_full_path": str(GENERATED_DATASETS_ROOT / "video_events_simulated_realistic.csv"),
    "realistic_video_filename": "video_events_simulated_realistic.csv",
    "force_regenerate_video_dataset": True,
    "video_n_cameras": 10,
    "video_n_zones": 5,
    "video_days": 60,
    "video_event_freq": "2min",
    "video_hidden_incident_rate": 0.18,
    "video_correlated_burst_prob": 0.08,
    "video_label_flip_prob": 0.04,
    "video_test_size": 0.30,
    "video_use_chronological_split": True,

    # Optional caps
    "max_cyber_rows": None,
    "max_video_rows": 120000,

    # Branch models
    "models_to_run": ["LogisticRegression", "CalibratedLogisticRegression", "RandomForest", "ExtraTrees"],
    "n_estimators": 200,
    "max_depth": 12,
    "min_samples_leaf": 8,
    "class_weight": "balanced_subsample",

    # Fusion fixes
    "force_fusion_cyber_model": "CalibratedLogisticRegression",
    "force_fusion_video_model": None,
    "fusion_window_seconds": 120,
    "cyber_alert_threshold": 0.70,
    "video_alert_threshold": 0.65,
    "window_high_risk_threshold": 0.45,
    "window_uncertainty_threshold": 0.45,
    "fusion_uncertainty_penalty": 0.30,
    "fusion_confidence_cap": 0.80,

    # Graph fixes
    "graph_similarity_threshold": 0.92,
    "graph_max_similarity_sample": 5000,
    "graph_temporal_edge_seconds": 60,
}

random.seed(CONFIG["random_state"])
np.random.seed(CONFIG["random_state"])

log_section("Configuration")
for key in sorted(CONFIG):
    log_kv(key, CONFIG[key])

save_json(CONFIG, "config.json")


Configuration
class_weight: balanced_subsample
cyber_alert_threshold: 0.7
cyber_context_col: type
cyber_filename: ton_iot.csv
cyber_full_path: None
cyber_subdir: Datasets/TON_IoT
cyber_target_col: label
cyber_test_size: 0.3
force_fusion_cyber_model: CalibratedLogisticRegression
force_fusion_video_model: None
force_regenerate_video_dataset: True
fusion_confidence_cap: 0.8
fusion_uncertainty_penalty: 0.3
fusion_window_seconds: 120
graph_max_similarity_sample: 5000
graph_similarity_threshold: 0.92
graph_temporal_edge_seconds: 60
max_cyber_rows: None
max_depth: 12
max_video_rows: 120000
min_samples_leaf: 8
models_to_run: ['LogisticRegression', 'CalibratedLogisticRegression', 'RandomForest', 'ExtraTrees']
n_estimators: 200
random_state: 42
realistic_video_filename: video_events_simulated_realistic.csv
realistic_video_full_path: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\generated_datasets\video_events_simulated_realistic.csv
strict_identifier_exclusion: True
video_alert_threshold: 0.65
vide

WindowsPath('F:/MBZUAI UAE/7-CMES-SI/GitHub-v2/Outputs/20260413_161819/others/config.json')

## 4. Load and audit the cyber dataset

This cell loads the cyber dataset, removes duplicates before splitting, and saves the basic audit outputs.

In [12]:
log_section("Load cyber dataset")

cyber_path = resolve_existing_path(
    filename=CONFIG.get("cyber_filename"),
    subdir=CONFIG.get("cyber_subdir"),
    full_path=CONFIG.get("cyber_full_path"),
)

cyber_df = pd.read_csv(cyber_path)

log_kv("resolved_cyber_path", cyber_path)
log_kv("cyber_shape", cyber_df.shape)
log_kv("cyber_memory_mb", round(cyber_df.memory_usage(deep=True).sum() / (1024**2), 2))

log_section("Audit cyber dataset")

duplicate_count = int(cyber_df.duplicated().sum())
missing_total = int(cyber_df.isna().sum().sum())

log_kv("cyber_duplicates", duplicate_count)
log_kv("cyber_missing_total", missing_total)

audit_df = pd.DataFrame([{
    "rows_before": len(cyber_df),
    "columns_before": cyber_df.shape[1],
    "duplicates": duplicate_count,
    "missing_total": missing_total,
}])
save_csv(audit_df, "cyber_audit_summary.csv")

cyber_df = cyber_df.drop_duplicates().reset_index(drop=True)

if CONFIG["max_cyber_rows"] is not None:
    cyber_df = cyber_df.sample(CONFIG["max_cyber_rows"], random_state=CONFIG["random_state"]).reset_index(drop=True)

label_counts = cyber_df[CONFIG["cyber_target_col"]].value_counts().sort_index()
log_kv("cyber_shape_after_drop_duplicates", cyber_df.shape)
log_kv("cyber_label_distribution", label_counts.to_dict())

label_df = pd.DataFrame({"label": label_counts.index.astype(int), "count": label_counts.values})
save_csv(label_df, "cyber_label_distribution.csv")

plt.figure(figsize=(7, 5))
plt.bar(label_df["label"].astype(str), label_df["count"])
plt.title("Cyber label distribution after duplicate removal")
plt.xlabel("Cyber label")
plt.ylabel("Count")
save_figure("fig_cyber_label_distribution.png")


Load cyber dataset
resolved_cyber_path: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Datasets\TON_IoT\ton_iot.csv
cyber_shape: (211043, 44)
cyber_memory_mb: 350.24

Audit cyber dataset
cyber_duplicates: 20569
cyber_missing_total: 0
Saved table: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\tables\cyber_audit_summary.csv
cyber_shape_after_drop_duplicates: (190474, 44)
cyber_label_distribution: {0: 42040, 1: 148434}
Saved table: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\tables\cyber_label_distribution.csv
Saved figure: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\figures\fig_cyber_label_distribution.png


WindowsPath('F:/MBZUAI UAE/7-CMES-SI/GitHub-v2/Outputs/20260413_161819/figures/fig_cyber_label_distribution.png')

## 5. Create or load the rich synthetic video dataset

This cell generates a consistent video schema with `suspicious_label`, `hidden_incident`, uncertainty-related variables, and timestamped events.

In [13]:
def generate_realistic_video_events(config, rng):
    n_cameras = int(config.get("video_n_cameras", 10))
    n_zones = int(config.get("video_n_zones", 5))
    video_days = int(config.get("video_days", 60))
    freq = config.get("video_event_freq", "2min")

    start_time = pd.Timestamp("2024-01-01 00:00:00")
    freq_delta = pd.to_timedelta(freq)
    end_time = start_time + pd.Timedelta(days=video_days) - freq_delta

    timestamps = pd.date_range(start=start_time, end=end_time, freq=freq)

    n = len(timestamps)
    camera_ids = rng.integers(0, n_cameras, size=n)
    zone_ids = rng.integers(0, n_zones, size=n)

    hidden_incident = np.zeros(n, dtype=int)
    base_rate = float(config["video_hidden_incident_rate"])
    burst_prob = float(config["video_correlated_burst_prob"])

    for i in range(n):
        if i == 0:
            hidden_incident[i] = rng.binomial(1, base_rate)
        else:
            prev = hidden_incident[i - 1]
            p = min(max(base_rate + (burst_prob if prev == 1 else 0.0), 0.0), 0.95)
            hidden_incident[i] = rng.binomial(1, p)

    camera_risk = rng.normal(0.0, 0.35, size=n_cameras)
    zone_risk = rng.normal(0.0, 0.40, size=n_zones)

    motion_intensity = np.clip(
        0.35 + 0.35 * hidden_incident + camera_risk[camera_ids] * 0.15 + rng.normal(0, 0.18, size=n),
        0, 1
    )
    scene_density = np.clip(
        0.30 + 0.30 * hidden_incident + zone_risk[zone_ids] * 0.20 + rng.normal(0, 0.20, size=n),
        0, 1
    )
    authenticity_score = np.clip(0.65 - 0.15 * hidden_incident + rng.normal(0, 0.20, size=n), 0, 1)
    temporal_consistency = np.clip(0.60 - 0.10 * hidden_incident + rng.normal(0, 0.22, size=n), 0, 1)
    uncertainty_score = np.clip(0.35 + 0.20 * hidden_incident + rng.normal(0, 0.20, size=n), 0, 1)

    suspicious_logit = (
        1.2 * hidden_incident
        + 1.1 * motion_intensity
        + 0.9 * scene_density
        - 0.6 * authenticity_score
        - 0.4 * temporal_consistency
        + 0.6 * uncertainty_score
        + 0.20 * camera_risk[camera_ids]
        + 0.25 * zone_risk[zone_ids]
        + rng.normal(0, 0.65, size=n)
    )

    suspicious_probability = 1 / (1 + np.exp(-suspicious_logit))
    suspicious_label = (suspicious_probability > 0.58).astype(int)

    flip_mask = rng.random(n) < float(config["video_label_flip_prob"])
    suspicious_label[flip_mask] = 1 - suspicious_label[flip_mask]

    df = pd.DataFrame({
        "timestamp": timestamps,
        "camera_id": camera_ids,
        "zone_id": zone_ids,
        "motion_intensity": motion_intensity,
        "scene_density": scene_density,
        "authenticity_score": authenticity_score,
        "temporal_consistency": temporal_consistency,
        "uncertainty_score": uncertainty_score,
        "hidden_incident": hidden_incident,
        "suspicious_probability": suspicious_probability,
        "suspicious_label": suspicious_label,
    })

    meta = {
        "rows": int(len(df)),
        "n_cameras": n_cameras,
        "n_zones": n_zones,
        "start_timestamp": str(df["timestamp"].min()),
        "end_timestamp": str(df["timestamp"].max()),
        "label_distribution": df["suspicious_label"].value_counts().sort_index().to_dict(),
        "hidden_incident_distribution": df["hidden_incident"].value_counts().sort_index().to_dict(),
    }
    return df, meta

log_section("Create or load realistic video dataset")

video_path = Path(CONFIG["realistic_video_full_path"])
video_path.parent.mkdir(parents=True, exist_ok=True)

if video_path.exists() and not CONFIG["force_regenerate_video_dataset"]:
    video_df = pd.read_csv(video_path, parse_dates=["timestamp"])
    video_generation_meta = {"rows": int(len(video_df)), "loaded_existing": True}
else:
    log_line("Generating realistic video dataset from a hidden-state process...")
    rng = np.random.default_rng(CONFIG["random_state"])
    video_df, video_generation_meta = generate_realistic_video_events(CONFIG, rng)
    video_df.to_csv(video_path, index=False)

if CONFIG["max_video_rows"] is not None and len(video_df) > CONFIG["max_video_rows"]:
    video_df = video_df.iloc[:CONFIG["max_video_rows"]].copy()

log_kv("video_path", video_path)
log_kv("video_generation_meta", json.dumps(video_generation_meta, indent=2))
save_json(video_generation_meta, "video_generation_meta.json")


Create or load realistic video dataset
Generating realistic video dataset from a hidden-state process...
video_path: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\generated_datasets\video_events_simulated_realistic.csv
video_generation_meta: {
  "rows": 43200,
  "n_cameras": 10,
  "n_zones": 5,
  "start_timestamp": "2024-01-01 00:00:00",
  "end_timestamp": "2024-02-29 23:58:00",
  "label_distribution": {
    "0": 20577,
    "1": 22623
  },
  "hidden_incident_distribution": {
    "0": 34819,
    "1": 8381
  }
}
Saved json: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\others\video_generation_meta.json


WindowsPath('F:/MBZUAI UAE/7-CMES-SI/GitHub-v2/Outputs/20260413_161819/others/video_generation_meta.json')

## 6. Audit the video dataset and leakage diagnostics

This cell confirms the rich video schema and checks whether any single feature is trivially predictive.

In [14]:
log_section("Audit realistic video dataset")

video_duplicates = int(video_df.duplicated().sum())
video_missing_total = int(video_df.isna().sum().sum())
video_target_col = "suspicious_label"
video_hidden_col = "hidden_incident"

log_kv("video_shape", video_df.shape)
log_kv("video_duplicates", video_duplicates)
log_kv("video_missing_total", video_missing_total)
log_kv("video_target_col", video_target_col)
log_kv("video_label_distribution", video_df[video_target_col].value_counts().sort_index().to_dict())
log_kv("video_hidden_incident_distribution", video_df[video_hidden_col].value_counts().sort_index().to_dict())

save_csv(
    pd.DataFrame({
        "suspicious_label": video_df[video_target_col].value_counts().sort_index().index,
        "count": video_df[video_target_col].value_counts().sort_index().values
    }),
    "video_label_distribution.csv"
)

save_csv(
    pd.DataFrame({
        "hidden_incident": video_df[video_hidden_col].value_counts().sort_index().index,
        "count": video_df[video_hidden_col].value_counts().sort_index().values
    }),
    "video_hidden_incident_distribution.csv"
)

plt.figure(figsize=(7, 5))
video_df[video_target_col].value_counts().sort_index().plot(kind="bar")
plt.title("Video-event label distribution")
plt.xlabel("Suspicious label")
plt.ylabel("Count")
save_figure("fig_video_label_distribution.png")

plt.figure(figsize=(7, 5))
video_df[video_hidden_col].value_counts().sort_index().plot(kind="bar")
plt.title("Video hidden-incident distribution")
plt.xlabel("Hidden incident state")
plt.ylabel("Count")
save_figure("fig_video_hidden_incident_distribution.png")

log_section("Video leakage diagnostics")

leakage_features = [
    "suspicious_probability", "hidden_incident", "motion_intensity", "scene_density",
    "zone_id", "camera_id", "temporal_consistency", "uncertainty_score", "authenticity_score"
]
leakage_df = single_feature_auc_table(video_df, video_target_col, leakage_features)
save_csv(leakage_df, "video_leakage_diagnostics.csv")
display(leakage_df)

plt.figure(figsize=(9, 5))
plt.barh(leakage_df["feature"], leakage_df["roc_auc"])
plt.gca().invert_yaxis()
plt.title("Video leakage diagnostics")
plt.xlabel("Single-feature ROC-AUC")
save_figure("fig_video_leakage_diagnostics_roc_auc.png")


Audit realistic video dataset
video_shape: (43200, 11)
video_duplicates: 0
video_missing_total: 0
video_target_col: suspicious_label
video_label_distribution: {0: 20577, 1: 22623}
video_hidden_incident_distribution: {0: 34819, 1: 8381}
Saved table: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\tables\video_label_distribution.csv
Saved table: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\tables\video_hidden_incident_distribution.csv
Saved figure: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\figures\fig_video_label_distribution.png
Saved figure: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\figures\fig_video_hidden_incident_distribution.png

Video leakage diagnostics
Saved table: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\tables\video_leakage_diagnostics.csv


,feature,roc_auc
0,suspicious_probability,0.958862
2,motion_intensity,0.724739
3,scene_density,0.693335
1,hidden_incident,0.668389
7,uncertainty_score,0.638462
8,authenticity_score,0.621339
6,temporal_consistency,0.584631
5,camera_id,0.534077
4,zone_id,0.501775


Saved figure: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\figures\fig_video_leakage_diagnostics_roc_auc.png


WindowsPath('F:/MBZUAI UAE/7-CMES-SI/GitHub-v2/Outputs/20260413_161819/figures/fig_video_leakage_diagnostics_roc_auc.png')

## 7. Prepare cyber features with stronger leakage exclusion

This cell removes the target, context fields, identifiers, and also several quasi-identifier traffic-volume fields that were likely helping the models memorize flows.

In [15]:
log_section("Cyber feature preparation")

cyber_target_col = CONFIG["cyber_target_col"]
cyber_context_col = CONFIG["cyber_context_col"]

cyber_drop_cols = [cyber_target_col]
if cyber_context_col in cyber_df.columns:
    cyber_drop_cols.append(cyber_context_col)

identifier_like_cols = [
    "src_ip", "dst_ip", "http_uri", "ssl_subject", "ssl_issuer", "dns_query",
    "event_index", "timestamp", "id", "session_id"
]

quasi_identifier_cols = [
    "src_bytes", "dst_bytes", "src_pkts", "dst_pkts",
    "src_ip_bytes", "dst_ip_bytes"
]

if CONFIG["strict_identifier_exclusion"]:
    cyber_drop_cols.extend([c for c in identifier_like_cols if c in cyber_df.columns])
    cyber_drop_cols.extend([c for c in quasi_identifier_cols if c in cyber_df.columns])

cyber_drop_cols = sorted(set([c for c in cyber_drop_cols if c in cyber_df.columns]))

Xc = cyber_df.drop(columns=cyber_drop_cols).copy()
yc = cyber_df[cyber_target_col].astype(int).copy()

log_kv("cyber_feature_shape", Xc.shape)

cyber_numeric_features = Xc.select_dtypes(include=np.number).columns.tolist()
cyber_categorical_features = [c for c in Xc.columns if c not in cyber_numeric_features]

log_kv("cyber_numeric_features", len(cyber_numeric_features))
log_kv("cyber_categorical_features", len(cyber_categorical_features))
log_kv("cyber_dropped_columns", cyber_drop_cols)


Cyber feature preparation
cyber_feature_shape: (190474, 30)
cyber_numeric_features: 10
cyber_categorical_features: 20
cyber_dropped_columns: ['dns_query', 'dst_bytes', 'dst_ip', 'dst_ip_bytes', 'dst_pkts', 'http_uri', 'label', 'src_bytes', 'src_ip', 'src_ip_bytes', 'src_pkts', 'ssl_issuer', 'ssl_subject', 'type']


## 8. Prepare video features

This cell expands timestamps and creates both the full and ablation feature sets.

In [16]:
log_section("Video feature preparation")

video_df = video_df.sort_values("timestamp").reset_index(drop=True)
video_df["hour"] = video_df["timestamp"].dt.hour
video_df["dayofweek"] = video_df["timestamp"].dt.dayofweek
video_df["dayofyear"] = video_df["timestamp"].dt.dayofyear
video_df["unix_time"] = (video_df["timestamp"].astype("int64") // 10**9).astype(np.int64)

video_target_col = "suspicious_label"

video_full_feature_cols = [
    "camera_id", "zone_id", "motion_intensity", "scene_density",
    "authenticity_score", "temporal_consistency", "uncertainty_score",
    "hour", "dayofweek", "dayofyear", "unix_time", "hidden_incident"
]
video_ablation_feature_cols = [
    "camera_id", "zone_id", "motion_intensity", "scene_density",
    "hour", "dayofweek", "dayofyear", "unix_time", "hidden_incident"
]

Xv_full = video_df[video_full_feature_cols].copy()
Xv_ablation = video_df[video_ablation_feature_cols].copy()
yv = video_df[video_target_col].astype(int).copy()

log_kv("video_feature_shape", Xv_full.shape)
log_kv("video_ablation_shape", Xv_ablation.shape)


Video feature preparation
video_feature_shape: (43200, 12)
video_ablation_shape: (43200, 9)


## 9. Split the cyber and video branches

This cell uses stratified splitting for cyber and chronological splitting for video.

In [17]:
log_section("Cyber split summary")

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=CONFIG["cyber_test_size"], stratify=yc, random_state=CONFIG["random_state"]
)

log_kv("Xc_train_shape", Xc_train.shape)
log_kv("Xc_test_shape", Xc_test.shape)
log_kv("cyber_numeric_features", len(cyber_numeric_features))
log_kv("cyber_categorical_features", len(cyber_categorical_features))

log_section("Video split summary")

split_idx = int((1 - CONFIG["video_test_size"]) * len(video_df))
Xv_train = Xv_full.iloc[:split_idx].copy()
Xv_test = Xv_full.iloc[split_idx:].copy()
Xv_train_ab = Xv_ablation.iloc[:split_idx].copy()
Xv_test_ab = Xv_ablation.iloc[split_idx:].copy()
yv_train = yv.iloc[:split_idx].copy()
yv_test = yv.iloc[split_idx:].copy()
video_train_meta = video_df.iloc[:split_idx].copy()
video_test_meta = video_df.iloc[split_idx:].copy()

log_kv("Xv_train_shape", Xv_train.shape)
log_kv("Xv_test_shape", Xv_test.shape)
log_kv("video_train_start", video_train_meta["timestamp"].min())
log_kv("video_train_end", video_train_meta["timestamp"].max())
log_kv("video_test_start", video_test_meta["timestamp"].min())
log_kv("video_test_end", video_test_meta["timestamp"].max())


Cyber split summary
Xc_train_shape: (133331, 30)
Xc_test_shape: (57143, 30)
cyber_numeric_features: 10
cyber_categorical_features: 20

Video split summary
Xv_train_shape: (30239, 12)
Xv_test_shape: (12961, 12)
video_train_start: 2024-01-01 00:00:00
video_train_end: 2024-02-11 23:56:00
video_test_start: 2024-02-11 23:58:00
video_test_end: 2024-02-29 23:58:00


## 10. Define preprocessing and candidate models

This cell defines the branch-specific preprocessing pipelines and model factory.

In [18]:
def make_cyber_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), cyber_numeric_features),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), cyber_categorical_features)
        ],
        remainder="drop"
    )

def make_video_preprocessor(feature_cols):
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), feature_cols)
        ],
        remainder="drop"
    )

def make_model(name):
    if name == "LogisticRegression":
        return LogisticRegression(max_iter=1000, random_state=CONFIG["random_state"])
    elif name == "CalibratedLogisticRegression":
        base = LogisticRegression(max_iter=1000, random_state=CONFIG["random_state"])
        return CalibratedClassifierCV(base, method="sigmoid", cv=3)
    elif name == "RandomForest":
        return RandomForestClassifier(
            n_estimators=CONFIG["n_estimators"],
            max_depth=CONFIG["max_depth"],
            min_samples_leaf=CONFIG["min_samples_leaf"],
            class_weight=CONFIG["class_weight"],
            random_state=CONFIG["random_state"],
            n_jobs=-1,
        )
    elif name == "ExtraTrees":
        return ExtraTreesClassifier(
            n_estimators=CONFIG["n_estimators"],
            max_depth=CONFIG["max_depth"],
            min_samples_leaf=CONFIG["min_samples_leaf"],
            class_weight=CONFIG["class_weight"],
            random_state=CONFIG["random_state"],
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Unknown model: {name}")

## 11. Train and evaluate cyber models

This cell trains the cyber branch models and saves a comparison table and figure.

In [19]:
log_section("Train cyber models")

cyber_results = []
cyber_fitted = {}

for model_name in CONFIG["models_to_run"]:
    pipe = Pipeline([
        ("preprocessor", make_cyber_preprocessor()),
        ("model", make_model(model_name))
    ])
    pipe.fit(Xc_train, yc_train)
    yc_pred = pipe.predict(Xc_test)
    yc_score = pipe.predict_proba(Xc_test)[:, 1] if hasattr(pipe, "predict_proba") else yc_pred.astype(float)

    metrics = compute_metrics(yc_test, yc_pred, yc_score)
    metrics["model"] = model_name
    cyber_results.append(metrics)
    cyber_fitted[model_name] = pipe
    log_kv(f"cyber_{model_name}", metrics)

cyber_results_df = pd.DataFrame(cyber_results).sort_values("balanced_accuracy", ascending=False)
save_csv(cyber_results_df, "cyber_model_results.csv")
display(cyber_results_df)

plt.figure(figsize=(10, 5))
cyber_results_df.set_index("model")[["accuracy", "balanced_accuracy", "roc_auc", "pr_auc"]].plot(kind="bar", rot=90)
plt.title("Cyber model comparison")
plt.ylabel("Score")
plt.xlabel("model")
save_figure("fig_cyber_model_comparison.png")


Train cyber models
cyber_LogisticRegression: {'accuracy': 0.9850550373624066, 'balanced_accuracy': 0.979300264269588, 'precision': 0.9912053802379721, 'recall': 0.9896027486470099, 'f1': 0.9904034161141702, 'roc_auc': 0.9923911324940159, 'pr_auc': 0.9973867895226883, 'fpr': 0.03100222010783381, 'tp': 44068, 'tn': 12221, 'fp': 391, 'fn': 463, 'model': 'LogisticRegression'}
cyber_CalibratedLogisticRegression: {'accuracy': 0.9850725373186567, 'balanced_accuracy': 0.9792830757532627, 'precision': 0.9911834825244029, 'recall': 0.9896476611798523, 'f1': 0.9904149764588226, 'roc_auc': 0.9923191324904264, 'pr_auc': 0.997353133881103, 'fpr': 0.03108150967332699, 'tp': 44070, 'tn': 12220, 'fp': 392, 'fn': 461, 'model': 'CalibratedLogisticRegression'}
cyber_RandomForest: {'accuracy': 0.9947675130812172, 'balanced_accuracy': 0.9926076239359243, 'precision': 0.9968101356815527, 'recall': 0.9964743661718802, 'f1': 0.9966422226464676, 'roc_auc': 0.9989347508928076, 'pr_auc': 0.9997020753819104, 'fpr

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,fpr,tp,tn,fp,fn,model
2,0.994768,0.992608,0.996810,0.996474,0.996642,0.998935,0.999702,0.011259,44374,12470,142,157,RandomForest
3,0.985143,0.981090,0.992558,0.988345,0.990447,0.996266,0.998895,0.026166,44012,12282,330,519,ExtraTrees
0,0.985055,0.979300,0.991205,0.989603,0.990403,0.992391,0.997387,0.031002,44068,12221,391,463,LogisticRegression
1,0.985073,0.979283,0.991183,0.989648,0.990415,0.992319,0.997353,0.031082,44070,12220,392,461,CalibratedLogisticRegression


Saved figure: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\figures\fig_cyber_model_comparison.png


WindowsPath('F:/MBZUAI UAE/7-CMES-SI/GitHub-v2/Outputs/20260413_161819/figures/fig_cyber_model_comparison.png')

<Figure size 1000x500 with 0 Axes>

## 12. Train and evaluate video models

This cell trains the video branch models and saves a comparison table and figure.

In [20]:
log_section("Train video models")

video_results = []
video_fitted = {}

for model_name in ["LogisticRegression", "RandomForest", "ExtraTrees"]:
    pipe = Pipeline([
        ("preprocessor", make_video_preprocessor(video_full_feature_cols)),
        ("model", make_model(model_name))
    ])
    pipe.fit(Xv_train, yv_train)
    yv_pred = pipe.predict(Xv_test)
    yv_score = pipe.predict_proba(Xv_test)[:, 1] if hasattr(pipe, "predict_proba") else yv_pred.astype(float)

    metrics = compute_metrics(yv_test, yv_pred, yv_score)
    metrics["model"] = model_name
    video_results.append(metrics)
    video_fitted[model_name] = pipe
    log_kv(f"video_{model_name}", metrics)

video_results_df = pd.DataFrame(video_results).sort_values("balanced_accuracy", ascending=False)
save_csv(video_results_df, "video_model_results.csv")
display(video_results_df)

plt.figure(figsize=(10, 5))
video_results_df.set_index("model")[["accuracy", "balanced_accuracy", "roc_auc", "pr_auc"]].plot(kind="bar", rot=90)
plt.title("Video model comparison")
plt.ylabel("Score")
plt.xlabel("model")
save_figure("fig_video_model_comparison.png")


Train video models
video_LogisticRegression: {'accuracy': 0.7094360003086182, 'balanced_accuracy': 0.7141369750440137, 'precision': 0.7763275522975147, 'recall': 0.6332215254484469, 'f1': 0.6975100401606426, 'roc_auc': 0.7950874263244398, 'pr_auc': 0.828760360571893, 'fpr': 0.2049475753604194, 'tp': 4342, 'tn': 4853, 'fp': 1251, 'fn': 2515, 'model': 'LogisticRegression'}
video_RandomForest: {'accuracy': 0.7085873003626263, 'balanced_accuracy': 0.7148910642442665, 'precision': 0.7941176470588235, 'recall': 0.6063876330756891, 'f1': 0.6876705532126023, 'roc_auc': 0.7923314677236204, 'pr_auc': 0.8290560829237907, 'fpr': 0.17660550458715596, 'tp': 4158, 'tn': 5026, 'fp': 1078, 'fn': 2699, 'model': 'RandomForest'}
video_ExtraTrees: {'accuracy': 0.7033407916055859, 'balanced_accuracy': 0.7138455890040523, 'precision': 0.8503955328059563, 'recall': 0.5330319381653784, 'f1': 0.6553115194979829, 'roc_auc': 0.7969851388341234, 'pr_auc': 0.8317825651359412, 'fpr': 0.10534076015727392, 'tp': 3655

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,fpr,tp,tn,fp,fn,model
1,0.708587,0.714891,0.794118,0.606388,0.687671,0.792331,0.829056,0.176606,4158,5026,1078,2699,RandomForest
0,0.709436,0.714137,0.776328,0.633222,0.697510,0.795087,0.828760,0.204948,4342,4853,1251,2515,LogisticRegression
2,0.703341,0.713846,0.850396,0.533032,0.655312,0.796985,0.831783,0.105341,3655,5461,643,3202,ExtraTrees


Saved figure: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\figures\fig_video_model_comparison.png


WindowsPath('F:/MBZUAI UAE/7-CMES-SI/GitHub-v2/Outputs/20260413_161819/figures/fig_video_model_comparison.png')

<Figure size 1000x500 with 0 Axes>

## 13. Video ablation study

This cell compares full vs. ablated video models.

In [21]:
log_section("Video ablation study")

video_ablation_results = []

for model_name in ["LogisticRegression", "RandomForest", "ExtraTrees"]:
    pipe = Pipeline([
        ("preprocessor", make_video_preprocessor(video_ablation_feature_cols)),
        ("model", make_model(model_name))
    ])
    pipe.fit(Xv_train_ab, yv_train)
    yv_pred = pipe.predict(Xv_test_ab)
    yv_score = pipe.predict_proba(Xv_test_ab)[:, 1] if hasattr(pipe, "predict_proba") else yv_pred.astype(float)

    metrics = compute_metrics(yv_test, yv_pred, yv_score)
    metrics["model"] = model_name
    video_ablation_results.append(metrics)
    log_kv(f"video_ablation_{model_name}", metrics)

video_ablation_df = pd.DataFrame(video_ablation_results).sort_values("balanced_accuracy", ascending=False)
save_csv(video_ablation_df, "video_ablation_results.csv")
display(video_ablation_df)

comparison_df = video_results_df[["model", "balanced_accuracy"]].merge(
    video_ablation_df[["model", "balanced_accuracy"]],
    on="model",
    suffixes=("_full", "_ablation")
)
save_csv(comparison_df, "video_full_vs_ablation_comparison.csv")

plt.figure(figsize=(10, 5))
x = np.arange(len(comparison_df))
w = 0.35
plt.bar(x - w/2, comparison_df["balanced_accuracy_ablation"], width=w, label="Ablation")
plt.bar(x + w/2, comparison_df["balanced_accuracy_full"], width=w, label="Full")
plt.xticks(x, comparison_df["model"], rotation=90)
plt.ylabel("Balanced accuracy")
plt.xlabel("model")
plt.title("Video branch: full features vs. ablation")
plt.legend()
save_figure("fig_video_full_vs_ablation_balanced_accuracy.png")


Video ablation study
video_ablation_LogisticRegression: {'accuracy': 0.6977085101458221, 'balanced_accuracy': 0.7037010733786311, 'precision': 0.7774211818010195, 'recall': 0.600554178212046, 'f1': 0.6776369919368109, 'roc_auc': 0.7759543585674855, 'pr_auc': 0.8160542684745018, 'fpr': 0.19315203145478374, 'tp': 4118, 'tn': 4925, 'fp': 1179, 'fn': 2739, 'model': 'LogisticRegression'}
video_ablation_RandomForest: {'accuracy': 0.6943908649023995, 'balanced_accuracy': 0.7020678087521319, 'precision': 0.7943089430894309, 'recall': 0.5699285401779204, 'f1': 0.6636664685403753, 'roc_auc': 0.7738499569276194, 'pr_auc': 0.8151879319006368, 'fpr': 0.1657929226736566, 'tp': 3908, 'tn': 5092, 'fp': 1012, 'fn': 2949, 'model': 'RandomForest'}
video_ablation_ExtraTrees: {'accuracy': 0.6949309466862125, 'balanced_accuracy': 0.7037206408734433, 'precision': 0.8106141664883373, 'recall': 0.5524281755869914, 'f1': 0.6570685169124024, 'roc_auc': 0.7780242124692582, 'pr_auc': 0.819185166229311, 'fpr': 0.1

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,fpr,tp,tn,fp,fn,model
2,0.694931,0.703721,0.810614,0.552428,0.657069,0.778024,0.819185,0.144987,3788,5219,885,3069,ExtraTrees
0,0.697709,0.703701,0.777421,0.600554,0.677637,0.775954,0.816054,0.193152,4118,4925,1179,2739,LogisticRegression
1,0.694391,0.702068,0.794309,0.569929,0.663666,0.773850,0.815188,0.165793,3908,5092,1012,2949,RandomForest


Saved table: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\tables\video_full_vs_ablation_comparison.csv
Saved figure: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\figures\fig_video_full_vs_ablation_balanced_accuracy.png


WindowsPath('F:/MBZUAI UAE/7-CMES-SI/GitHub-v2/Outputs/20260413_161819/figures/fig_video_full_vs_ablation_balanced_accuracy.png')

## 14. Select branch models and force safer fusion models

This cell selects the best branch models for branch reporting, but it can force safer models for fusion. By default, fusion uses Calibrated Logistic Regression for the cyber branch.

In [22]:
log_section("Selected models")

best_cyber_model_name = cyber_results_df.iloc[0]["model"]
best_video_model_name = video_results_df.iloc[0]["model"]

best_cyber_model = cyber_fitted[best_cyber_model_name]
best_video_model = video_fitted[best_video_model_name]

fusion_cyber_model_name = CONFIG["force_fusion_cyber_model"] if CONFIG["force_fusion_cyber_model"] in cyber_fitted else best_cyber_model_name
fusion_video_model_name = CONFIG["force_fusion_video_model"] if CONFIG["force_fusion_video_model"] in video_fitted else best_video_model_name

fusion_cyber_model = cyber_fitted[fusion_cyber_model_name]
fusion_video_model = video_fitted[fusion_video_model_name]

log_kv("best_cyber_model", best_cyber_model_name)
log_kv("best_video_model", best_video_model_name)
log_kv("fusion_cyber_model", fusion_cyber_model_name)
log_kv("fusion_video_model", fusion_video_model_name)


Selected models
best_cyber_model: RandomForest
best_video_model: RandomForest
fusion_cyber_model: CalibratedLogisticRegression
fusion_video_model: RandomForest


## 15. Build timestamped test event tables

This cell creates timestamped cyber and video test-event tables for downstream fusion.

In [23]:
log_section("Test event tables")

cyber_test_scores = fusion_cyber_model.predict_proba(Xc_test)[:, 1]
cyber_test_pred = (cyber_test_scores >= CONFIG["cyber_alert_threshold"]).astype(int)

cyber_test_timestamps = pd.date_range(
    start="2024-02-12 00:00:00",
    periods=len(Xc_test),
    freq="20s"
)

cyber_test_events = pd.DataFrame({
    "timestamp": cyber_test_timestamps,
    "cyber_score": cyber_test_scores,
    "cyber_pred": cyber_test_pred,
    "cyber_label": yc_test.reset_index(drop=True),
})
cyber_test_events["cyber_uncertainty"] = 1 - 2 * np.abs(cyber_test_events["cyber_score"] - 0.5)

video_test_scores = fusion_video_model.predict_proba(Xv_test)[:, 1]
video_test_pred = (video_test_scores >= CONFIG["video_alert_threshold"]).astype(int)

video_test_events = video_test_meta[["timestamp", "camera_id", "zone_id"]].copy().reset_index(drop=True)
video_test_events["video_score"] = video_test_scores
video_test_events["video_pred"] = video_test_pred
video_test_events["video_label"] = yv_test.reset_index(drop=True)
video_test_events["video_uncertainty"] = 1 - 2 * np.abs(video_test_events["video_score"] - 0.5)

log_kv("cyber_test_events", cyber_test_events.shape)
log_kv("video_test_events", video_test_events.shape)

save_csv(cyber_test_events.head(500), "cyber_test_events_sample.csv")
save_csv(video_test_events.head(500), "video_test_events_sample.csv")


Test event tables
cyber_test_events: (57143, 5)
video_test_events: (12961, 7)
Saved table: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\tables\cyber_test_events_sample.csv
Saved table: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\tables\video_test_events_sample.csv


WindowsPath('F:/MBZUAI UAE/7-CMES-SI/GitHub-v2/Outputs/20260413_161819/tables/video_test_events_sample.csv')

## 16. Fuse cyber and video evidence with safer uncertainty handling

This cell applies capped confidence weights, a softer uncertainty penalty, and a lower decision threshold to avoid recall collapse.

In [24]:
log_section("Fusion window construction")

window_seconds = int(CONFIG["fusion_window_seconds"])

def assign_window_id(ts_series, origin=None, seconds=120):
    if origin is None:
        origin = ts_series.min()
    return ((ts_series - origin).dt.total_seconds() // seconds).astype(int)

fusion_origin = min(cyber_test_events["timestamp"].min(), video_test_events["timestamp"].min())

cyber_test_events["window_id"] = assign_window_id(cyber_test_events["timestamp"], fusion_origin, window_seconds)
video_test_events["window_id"] = assign_window_id(video_test_events["timestamp"], fusion_origin, window_seconds)

cyber_window = cyber_test_events.groupby("window_id").agg(
    cyber_mean_prob=("cyber_score", "mean"),
    cyber_mean_uncertainty=("cyber_uncertainty", "mean"),
    cyber_count=("cyber_score", "size"),
    cyber_label_any=("cyber_label", "max"),
).reset_index()

video_window = video_test_events.groupby("window_id").agg(
    video_mean_prob=("video_score", "mean"),
    video_mean_uncertainty=("video_uncertainty", "mean"),
    video_count=("video_score", "size"),
    video_label_any=("video_label", "max"),
).reset_index()

fusion_window_table = cyber_window.merge(video_window, on="window_id", how="outer").fillna(0)
fusion_window_table["modality_count"] = (
    (fusion_window_table["cyber_count"] > 0).astype(int) +
    (fusion_window_table["video_count"] > 0).astype(int)
)

cap = float(CONFIG["fusion_confidence_cap"])
cy_conf = np.minimum(1 - fusion_window_table["cyber_mean_uncertainty"], cap)
vi_conf = np.minimum(1 - fusion_window_table["video_mean_uncertainty"], cap)
conf_sum = (cy_conf + vi_conf).replace(0, 1e-9)

fusion_window_table["w_cyber"] = cy_conf / conf_sum
fusion_window_table["w_video"] = vi_conf / conf_sum

fusion_window_table["raw_fusion_score"] = (
    fusion_window_table["w_cyber"] * fusion_window_table["cyber_mean_prob"] +
    fusion_window_table["w_video"] * fusion_window_table["video_mean_prob"]
)

fusion_window_table["fusion_uncertainty"] = (
    fusion_window_table["w_cyber"] * fusion_window_table["cyber_mean_uncertainty"] +
    fusion_window_table["w_video"] * fusion_window_table["video_mean_uncertainty"]
)

penalty = float(CONFIG["fusion_uncertainty_penalty"])
fusion_window_table["adjusted_fusion_score"] = (
    fusion_window_table["raw_fusion_score"] - penalty * fusion_window_table["fusion_uncertainty"]
).clip(lower=0, upper=1)

fusion_window_table["fused_label_or"] = (
    (fusion_window_table["cyber_label_any"] > 0) | (fusion_window_table["video_label_any"] > 0)
).astype(int)

fusion_window_table["fused_pred"] = (
    fusion_window_table["adjusted_fusion_score"] >= CONFIG["window_high_risk_threshold"]
).astype(int)

def assign_window_state(row):
    if row["modality_count"] == 1:
        return "single_modality_only"
    if row["adjusted_fusion_score"] >= CONFIG["window_high_risk_threshold"]:
        return "high_risk"
    if row["fusion_uncertainty"] >= CONFIG["window_uncertainty_threshold"]:
        return "uncertain"
    if row["adjusted_fusion_score"] > 0:
        return "low_risk"
    return "benign"

fusion_window_table["window_state"] = fusion_window_table.apply(assign_window_state, axis=1)

save_csv(fusion_window_table, "fusion_window_table.csv")
display(fusion_window_table.head(20))


Fusion window construction
Saved table: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\tables\fusion_window_table.csv


,window_id,cyber_mean_prob,cyber_mean_uncertainty,cyber_count,cyber_label_any,video_mean_prob,video_mean_uncertainty,video_count,video_label_any,modality_count,w_cyber,w_video,raw_fusion_score,fusion_uncertainty,adjusted_fusion_score,fused_label_or,fused_pred,window_state
0,0,0.000000,0.000000,0.0,0.0,0.308631,0.617263,1,0,1,0.676397,0.323603,0.099874,0.199748,0.039950,0,0,single_modality_only
1,1,0.821448,0.023830,6.0,1.0,0.907460,0.185081,1,1,2,0.500000,0.500000,0.864454,0.104455,0.833117,1,1,high_risk
2,2,0.611049,0.183243,6.0,1.0,0.906492,0.187016,1,1,2,0.500000,0.500000,0.758770,0.185129,0.703231,1,1,high_risk
3,3,0.997277,0.005446,6.0,1.0,0.325811,0.651623,1,0,2,0.696635,0.303365,0.793578,0.201473,0.733136,1,1,high_risk
4,4,0.853613,0.045736,6.0,1.0,0.179582,0.359164,1,0,2,0.555233,0.444767,0.553826,0.185138,0.498285,1,1,high_risk
5,5,0.999804,0.000392,6.0,1.0,0.169817,0.339635,1,0,2,0.547808,0.452192,0.624491,0.153795,0.578352,1,1,high_risk
6,6,0.996065,0.007871,6.0,1.0,0.434651,0.869302,1,1,2,0.859570,0.140430,0.917225,0.128842,0.878573,1,1,high_risk
7,7,0.996721,0.006558,6.0,1.0,0.302797,0.605594,1,0,2,0.669789,0.330211,0.767580,0.204366,0.706270,1,1,high_risk
8,8,0.981013,0.037974,6.0,1.0,0.629675,0.740649,1,1,2,0.755180,0.244820,0.894999,0.210003,0.831998,1,1,high_risk
9,9,0.540291,0.088615,6.0,1.0,0.315025,0.630050,1,0,2,0.683790,0.316210,0.469059,0.259822,0.391113,1,0,low_risk


## 17. Evaluate fusion

This cell computes fusion metrics and saves the score-distribution and window-state figures.

In [25]:
log_section("Fusion metrics")

fusion_metrics = compute_metrics(
    fusion_window_table["fused_label_or"].astype(int),
    fusion_window_table["fused_pred"].astype(int),
    fusion_window_table["adjusted_fusion_score"].astype(float)
)

log_kv("fusion_metrics", fusion_metrics)
save_json(fusion_metrics, "fusion_metrics.json")

plt.figure(figsize=(9, 5))
plt.hist(fusion_window_table["adjusted_fusion_score"], bins=30)
plt.title("Distribution of adjusted fusion scores")
plt.xlabel("Adjusted fusion score")
plt.ylabel("Window count")
save_figure("fig_fusion_adjusted_score_distribution.png")

state_counts = fusion_window_table["window_state"].value_counts().rename_axis("window_state").reset_index(name="count")
save_csv(state_counts, "fusion_window_state_counts.csv")

plt.figure(figsize=(9, 5))
plt.bar(state_counts["window_state"], state_counts["count"])
plt.title("Fusion window states")
plt.xlabel("Window state")
plt.ylabel("Count")
plt.xticks(rotation=25)
save_figure("fig_fusion_window_states.png")


Fusion metrics
fusion_metrics: {'accuracy': 0.7885965589074917, 'balanced_accuracy': 0.8786411789770168, 'precision': 0.9998833819241982, 'recall': 0.7578891540705384, 'f1': 0.8622284794851166, 'roc_auc': 0.948952317936492, 'pr_auc': 0.9930247683399942, 'fpr': 0.0006067961165048543, 'tp': 8574, 'tn': 1647, 'fp': 1, 'fn': 2739}
Saved json: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\others\fusion_metrics.json
Saved figure: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\figures\fig_fusion_adjusted_score_distribution.png
Saved table: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\tables\fusion_window_state_counts.csv
Saved figure: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\figures\fig_fusion_window_states.png


WindowsPath('F:/MBZUAI UAE/7-CMES-SI/GitHub-v2/Outputs/20260413_161819/figures/fig_fusion_window_states.png')

## 18. Build a richer heterogeneous temporal graph

This cell adds window edges, similarity edges, and short-range temporal edges to reduce graph fragmentation.

In [26]:
log_section("Build heterogeneous temporal graph")

G = nx.Graph()

cyber_alerts = cyber_test_events[cyber_test_events["cyber_pred"] == 1].copy().reset_index(drop=True)
video_alerts = video_test_events[video_test_events["video_pred"] == 1].copy().reset_index(drop=True)

for i, row in cyber_alerts.iterrows():
    G.add_node(
        f"c_{i}",
        modality="cyber",
        timestamp=row["timestamp"],
        score=float(row["cyber_score"]),
        uncertainty=float(row["cyber_uncertainty"]),
        window_id=int(row["window_id"]),
    )

for i, row in video_alerts.iterrows():
    G.add_node(
        f"v_{i}",
        modality="video",
        timestamp=row["timestamp"],
        score=float(row["video_score"]),
        uncertainty=float(row["video_uncertainty"]),
        window_id=int(row["window_id"]),
    )

hub_windows = fusion_window_table[fusion_window_table["modality_count"] >= 1]["window_id"].tolist()
for wid in hub_windows:
    hub_row = fusion_window_table.loc[fusion_window_table["window_id"] == wid].iloc[0]
    G.add_node(
        f"h_{wid}",
        modality="hub",
        score=float(hub_row["adjusted_fusion_score"]),
        uncertainty=float(hub_row["fusion_uncertainty"]),
        window_state=hub_row["window_state"],
        window_id=int(wid),
    )

for i, row in cyber_alerts.iterrows():
    hub_id = f"h_{int(row['window_id'])}"
    if hub_id in G:
        G.add_edge(f"c_{i}", hub_id, edge_type="window")

for i, row in video_alerts.iterrows():
    hub_id = f"h_{int(row['window_id'])}"
    if hub_id in G:
        G.add_edge(f"v_{i}", hub_id, edge_type="window")

sim_threshold = CONFIG["graph_similarity_threshold"]
max_sim_sample = CONFIG["graph_max_similarity_sample"]

cyber_idx = cyber_alerts.index[:min(len(cyber_alerts), max_sim_sample)]
video_idx = video_alerts.index[:min(len(video_alerts), max_sim_sample)]

if len(cyber_idx) > 1:
    sim = cosine_similarity(cyber_alerts.loc[cyber_idx, ["cyber_score", "cyber_uncertainty"]].values)
    for i in range(len(cyber_idx)):
        for j in range(i + 1, len(cyber_idx)):
            if sim[i, j] >= sim_threshold:
                G.add_edge(f"c_{cyber_idx[i]}", f"c_{cyber_idx[j]}", edge_type="similarity")

if len(video_idx) > 1:
    sim = cosine_similarity(video_alerts.loc[video_idx, ["video_score", "video_uncertainty"]].values)
    for i in range(len(video_idx)):
        for j in range(i + 1, len(video_idx)):
            if sim[i, j] >= sim_threshold:
                G.add_edge(f"v_{video_idx[i]}", f"v_{video_idx[j]}", edge_type="similarity")

# Temporal edges within each modality
temporal_seconds = int(CONFIG["graph_temporal_edge_seconds"])

def add_temporal_edges(df, prefix):
    if len(df) < 2:
        return
    times = df["timestamp"].reset_index(drop=True)
    for i in range(len(df) - 1):
        dt = (times.iloc[i + 1] - times.iloc[i]).total_seconds()
        if dt <= temporal_seconds:
            G.add_edge(f"{prefix}_{i}", f"{prefix}_{i+1}", edge_type="temporal")

add_temporal_edges(cyber_alerts.sort_values("timestamp").reset_index(drop=True), "c")
add_temporal_edges(video_alerts.sort_values("timestamp").reset_index(drop=True), "v")

components = list(nx.connected_components(G))
graph_summary = {
    "graph_nodes": int(G.number_of_nodes()),
    "graph_edges": int(G.number_of_edges()),
    "connected_components": int(len(components)),
}
log_kv("graph_summary", graph_summary)
save_json(graph_summary, "graph_summary.json")

plt.figure(figsize=(8, 5))
plt.bar(["Nodes", "Edges", "Components"], [
    graph_summary["graph_nodes"],
    graph_summary["graph_edges"],
    graph_summary["connected_components"]
])
plt.title("Heterogeneous temporal graph summary")
save_figure("fig_graph_summary.png")


Build heterogeneous temporal graph
graph_summary: {'graph_nodes': 60507, 'graph_edges': 16108884, 'connected_components': 2669}
Saved json: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\others\graph_summary.json
Saved figure: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\figures\fig_graph_summary.png


WindowsPath('F:/MBZUAI UAE/7-CMES-SI/GitHub-v2/Outputs/20260413_161819/figures/fig_graph_summary.png')

## 19. Extract campaign candidates

This cell extracts connected components as campaign candidates and visualizes the campaign-size distribution.

In [27]:
log_section("Campaign candidates")

campaign_rows = []
for cid, comp in enumerate(components):
    sub = G.subgraph(comp)
    node_types = [G.nodes[n]["modality"] for n in sub.nodes]
    scores = [G.nodes[n].get("score", np.nan) for n in sub.nodes if "score" in G.nodes[n]]
    campaign_rows.append({
        "campaign_id": cid,
        "nodes": sub.number_of_nodes(),
        "edges": sub.number_of_edges(),
        "cyber_nodes": int(sum(t == "cyber" for t in node_types)),
        "video_nodes": int(sum(t == "video" for t in node_types)),
        "hub_nodes": int(sum(t == "hub" for t in node_types)),
        "mean_score": float(np.nanmean(scores)) if len(scores) else np.nan,
        "max_score": float(np.nanmax(scores)) if len(scores) else np.nan,
    })

campaign_df = pd.DataFrame(campaign_rows).sort_values(["nodes", "max_score"], ascending=[False, False])
save_csv(campaign_df, "campaign_candidates.csv")
display(campaign_df.head(20))

max_nodes = int(campaign_df["nodes"].max()) if len(campaign_df) else 1
plt.figure(figsize=(8, 5))
plt.hist(campaign_df["nodes"], bins=np.arange(1, max_nodes + 2) - 0.5)
plt.title("Distribution of campaign sizes")
plt.xlabel("Campaign size (nodes)")
plt.ylabel("Count")
save_figure("fig_campaign_size_distribution.png")


Campaign candidates
Saved table: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\tables\campaign_candidates.csv


,campaign_id,nodes,edges,cyber_nodes,video_nodes,hub_nodes,mean_score,max_score
0,0,57139,16107736,43790,3163,10186,0.920028,1.000000
26,26,53,87,44,0,9,0.926648,0.999999
21,21,52,83,42,0,10,0.914058,0.999999
23,23,46,75,38,0,8,0.924023,0.999999
1,1,43,69,35,0,8,0.916453,0.999999
10,10,41,65,33,0,8,0.902928,0.999999
20,20,38,61,31,0,7,0.915686,0.999999
7,7,36,56,29,0,7,0.909299,0.999999
4,4,35,55,28,0,7,0.889301,0.999999
32,32,34,55,28,0,6,0.912636,0.999998


Saved figure: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\figures\fig_campaign_size_distribution.png


WindowsPath('F:/MBZUAI UAE/7-CMES-SI/GitHub-v2/Outputs/20260413_161819/figures/fig_campaign_size_distribution.png')

In [30]:
print(campaign_df["nodes"].describe())
print(campaign_df["nodes"].sort_values(ascending=False).head(10))

count     2669.000000
mean        22.670288
std       1105.987536
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max      57139.000000
Name: nodes, dtype: float64
0     57139
26       53
21       52
23       46
1        43
10       41
20       38
7        36
4        35
32       34
Name: nodes, dtype: int64


In [34]:
plot_df = campaign_df[(campaign_df["nodes"] > 1) & (campaign_df["nodes"] < 200)]

plt.figure(figsize=(8, 5))
plt.hist(plot_df["nodes"], bins=30)
plt.title("Distribution of campaign sizes (2 to 199 nodes)")
plt.xlabel("Campaign size (nodes)")
plt.ylabel("Count")
save_figure("fig_campaign_size_distribution.png")

Saved figure: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\figures\fig_campaign_size_distribution.png


WindowsPath('F:/MBZUAI UAE/7-CMES-SI/GitHub-v2/Outputs/20260413_161819/figures/fig_campaign_size_distribution.png')

## 20. Final summary

This cell saves a concise summary for reproducibility and later paper writing.

In [28]:
log_section("Final summary")

final_summary = {
    "cyber_dataset_path": str(cyber_path),
    "video_dataset_path": str(video_path),
    "run_directory": str(RUN_DIR),
    "best_cyber_model": best_cyber_model_name,
    "best_video_model": best_video_model_name,
    "fusion_cyber_model": fusion_cyber_model_name,
    "fusion_video_model": fusion_video_model_name,
    "cyber_results_top": cyber_results_df.iloc[0].to_dict(),
    "video_results_top": video_results_df.iloc[0].to_dict(),
    "fusion_metrics": fusion_metrics,
    "graph_summary": graph_summary,
}

log_kv("Cyber dataset path", cyber_path)
log_kv("Video dataset path", video_path)
log_kv("Run directory", RUN_DIR)
log_kv("Best cyber model", best_cyber_model_name)
log_kv("Best video model", best_video_model_name)
log_kv("Fusion cyber model", fusion_cyber_model_name)
log_kv("Fusion video model", fusion_video_model_name)
log_kv("saved_summary_json", save_json(final_summary, "run_summary.json"))
log_kv("outputs_summary_path", SUMMARY_PATH)


Final summary
Cyber dataset path: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Datasets\TON_IoT\ton_iot.csv
Video dataset path: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\generated_datasets\video_events_simulated_realistic.csv
Run directory: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819
Best cyber model: RandomForest
Best video model: RandomForest
Fusion cyber model: CalibratedLogisticRegression
Fusion video model: RandomForest
Saved json: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\others\run_summary.json
saved_summary_json: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\20260413_161819\others\run_summary.json
outputs_summary_path: F:\MBZUAI UAE\7-CMES-SI\GitHub-v2\Outputs\outputs_summary.txt
